In [1]:
import pandas as pd
import numpy as np
import experimentation as exp
from contextlib import redirect_stdout, redirect_stderr
import io

In [2]:
def get_Data(csv):

    experiment_columns = [
        # 'Impedance',
        # 'ImpedancePhase',
        # 'Resistance',
        # 'Reactance',
        # 'Admittance',
        # 'AdmittancePhase',
        # 'Conductance',
        # 'Susceptance', 
        # 'SeriesCapacitance',
        # 'ParallelCapacitance',
        # 'SeriesInductance',
        # 'ParallelInductance',
        # # 'Dissipation',
        # 'Quality',
        # 'Vrms',
        'Vreal',
        'Vimag',
        # 'Irms',
        'Ireal',
        'Iimag'
    ]

    y_columns = [
        'Label',
        'Index'
    ]


    df = pd.read_csv(csv)

    features = []
    temp = []
    modified_features = []
    modified_temp = []
    max_iter = df["Iteration"].max()
    y = []
    for i in range(max_iter):
        temp = []
        modified_temp = []
        for col in df.columns:
            if col in experiment_columns:
                temp.extend(df[df["Iteration"] == i][col])
                modified_temp.append(df[df["Iteration"] == i][col].std())
                modified_temp.append(df[df["Iteration"] == i][col].mean())
                modified_temp.append(df[df["Iteration"] == i][col].median())
                modified_temp.append(df[df["Iteration"] == i][col].min())
                modified_temp.append(df[df["Iteration"] == i][col].max())

        features.append(temp)
        modified_features.append(modified_temp)
        data = df[df["Iteration"] == i][y_columns].values[0]
        y.append(data)
        # y.append(data / data.mean())

    features = np.array(features)
    modified_features = np.array(modified_features)
    y = np.array(y)

    print(features.shape)
    print(modified_features.shape)
    print(y.shape)

    return features, modified_features, y

In [3]:
path = './experiment_logs/'
file = 'merged_classification.csv'
file_path = path + file
x, xm ,y = get_Data(file_path)

iter_count = 50

(1097, 804)
(1097, 20)
(1097, 2)


In [4]:
index = y[:,1]
y = y[:,0]

In [5]:
labels = np.unique(y)
print(labels)

# convert char labels to numeric labels
y_numeric = np.zeros_like(y)
for i, label in enumerate(labels):
    y_numeric[y == label] = i

['A' 'B' 'C' 'D']


In [6]:
np.unique(y)

array(['A', 'B', 'C', 'D'], dtype=object)

In [7]:
y

array(['A', 'A', 'A', ..., 'D', 'D', 'D'], shape=(1097,), dtype=object)

In [8]:
df_x = pd.DataFrame(x)
df_x["label"] = y
df_x["index"] = index

df_xm = pd.DataFrame(xm)
df_xm["label"] = y
df_xm["index"] = index

x_grouped = df_x.groupby(["index", "label"])
xm_grouped = df_xm.groupby(["index", "label"])


In [9]:
keys = []
x_groups = []
xm_groups = []
y_groups = []

for key, g in df_x.groupby(["index", "label"], sort=False):
    y_groups.append(g["label"].to_numpy())  # (index, label) -> label
    x_groups.append(g.drop(columns=["index", "label"]).to_numpy())  # (n_i, 804)

for key, g in df_xm.groupby(["index", "label"], sort=False):
    xm_groups.append(g.drop(columns=["index", "label"]).to_numpy())  # (n_i, 20)

In [10]:
print(f"shape of x_groups: {[g.shape for g in x_groups]}")
print(f"shape of xm_groups: {[g.shape for g in xm_groups]}")
print(f"shape of y_groups: {[g.shape for g in y_groups]}")
print(f"number of groups: {len(x_groups)}")

shape of x_groups: [(54, 804), (36, 804), (54, 804), (60, 804), (57, 804), (59, 804), (53, 804), (88, 804), (70, 804), (53, 804), (54, 804), (53, 804), (56, 804), (109, 804), (54, 804), (54, 804), (78, 804), (55, 804)]
shape of xm_groups: [(54, 20), (36, 20), (54, 20), (60, 20), (57, 20), (59, 20), (53, 20), (88, 20), (70, 20), (53, 20), (54, 20), (53, 20), (56, 20), (109, 20), (54, 20), (54, 20), (78, 20), (55, 20)]
shape of y_groups: [(54,), (36,), (54,), (60,), (57,), (59,), (53,), (88,), (70,), (53,), (54,), (53,), (56,), (109,), (54,), (54,), (78,), (55,)]
number of groups: 18


In [12]:
print("starting",end="\r")
print("processing part 1",end="\r")
with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
    x_res = exp.eval(x_groups, y_groups, iteration_count=iter_count, regress=False, classify=True, split_by_grouped_data=True, save_file="iter_50/class/exp_orange_class_x_50_it.csv")
print("part 1 done               ")


part 1 done               


In [14]:
print("processing part 2",end="\r")
with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
    xm_res = exp.eval(xm_groups, y_groups, iteration_count=iter_count, regress=False, classify=True, split_by_grouped_data=True, save_file="iter_50/class/exp_orange_class_xm_50_it.csv")
print("part 2 done                ")

part 2 done                
